# Assignment 2, Hint 3: keep an average of the weights (EMA)

CSCI 6379 · companion to the [Assignment 2 hints page](https://dongchul.kim/csci6379-fall2026/assignment/2/hints).

Keep a **second copy** of the network whose weights are a slow running average of the trained ones: after every step, `average = 0.999 * average + 0.001 * current`. Gradient descent keeps training the ordinary network exactly as before; the average never feeds back into training. Every mini-batch nudges the weights a little at random, and the average cancels that jitter out over roughly the last 1,000 steps (20 epochs). You **validate, keep and submit the average**. In our runs, dropping this hint cost about 0.015 Dice (0.661 to 0.645), enough to fall below the A line.

**Where it goes.** `train_loop.py`, in three places, marked (1), (2) and (3) in the cell below, plus two small additions to the checkpoint code described under it.

This notebook holds **only the code of this hint**. It is not a training program and it
does not touch the assignment data. Copy the marked cell into your own Assignment 2
notebook, change `RUN_NAME` there, and train as before. The last cell is a small check
you can run here to see the hint working.

## Hint 3: the three pieces, shown inside a toy training loop

The loop below is **not** your loop: it trains a tiny network on random numbers so the cell runs here. Copy only the lines between the `HINT 3` markers into the same places in your `train_loop.py`.

In [ ]:
import copy
import torch, torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
net = nn.Sequential(nn.Linear(8, 16), nn.BatchNorm1d(16), nn.ReLU(), nn.Linear(16, 1))   # a stand-in network
opt = torch.optim.SGD(net.parameters(), lr=0.05)
X, y = torch.randn(512, 8), torch.randn(512, 1)

# ======== HINT 3 (1): right after the network, the optimizer and the scheduler are created ========
EMA = 0.999                       # how much of the old average to keep each step. 0.999 means the
                                  # average remembers about the last 1 / (1 - 0.999) = 1,000 steps.
ema = copy.deepcopy(net).eval()   # an independent copy with the same starting weights. .eval() because
                                  # it is never trained, only evaluated (batch norm uses its stored statistics).
# ======== end of HINT 3 (1) ========

raw_w, ema_w = [], []
for step in range(3000):
    net.train()
    idx = torch.randint(0, 512, (16,))                  # a random mini-batch, as in your loop
    loss = nn.functional.mse_loss(net(X[idx]), y[idx])
    opt.zero_grad(); loss.backward(); opt.step()        # gradient descent moves net, and only net
                                                        # (in your loop sched.step() comes next)

    # ======== HINT 3 (2): inside the batch loop, right after opt.step() and sched.step() ========
    with torch.no_grad():                               # bookkeeping, not learning: no gradients
        for pe, p in zip(ema.parameters(), net.parameters()):
            pe.lerp_(p, 1 - EMA)                        # pe = pe + 0.001 * (p - pe)
                                                        #    = 0.999 * pe + 0.001 * p
        for be, b in zip(ema.buffers(), net.buffers()):
            be.copy_(b)                                 # buffers are batch norm's running mean and
                                                        # variance: not learned by gradients, so they
                                                        # are copied, not averaged
    # ======== end of HINT 3 (2) ========

    raw_w.append(net[0].weight[0, 0].item()); ema_w.append(ema[0].weight[0, 0].item())

# ======== HINT 3 (3): after each epoch, measure and keep the EMA copy instead of net ========
# In your train_loop.py, change two existing lines so they use ema where they used net:
#     dice = val_dice(ema, Xva_t, Mva_t).mean().item()                                   # was val_dice(net, ...)
#     best_state = {k: v.detach().cpu().clone() for k, v in ema.state_dict().items()}    # was net.state_dict()
# The loop's last line, net.load_state_dict(best_state), then puts the best EMA weights into
# net, and weights.pth is written from them.
# ======== end of HINT 3 (3) ========

plt.figure(figsize=(7, 3))
plt.plot(raw_w, lw=0.6, color="gray", label="net (what gradient descent updates)")
plt.plot(ema_w, lw=2, color="tab:orange", label="ema (the running average)")
plt.xlabel("step"); plt.ylabel("one weight"); plt.legend(); plt.show()

## Two more lines for the checkpoints

So that a resumed run keeps its average, in `train_loop.py`:

- in the `torch.save({...})` dictionary, add `"ema": ema.state_dict(),`
- in the resume block, next to `opt.load_state_dict(ck["optimizer"])`, add `ema.load_state_dict(ck["ema"])`

An old checkpoint has no `"ema"` entry, which is one more reason to change `RUN_NAME`.